In [1]:
import duckdb
con = duckdb.connect("data/interim/olist.duckdb")

# 3단계 모집단: low=1 주문 + 리뷰 제목·본문 (master는 low NULL=리뷰 없음)
con.execute("""
CREATE OR REPLACE TABLE s3_pool AS
SELECT m.order_id, m.late, m.purchase_ts, r.review_id,
       r.review_comment_title AS title, r.review_comment_message AS msg
FROM master m JOIN rev r USING (order_id)
WHERE m.low = 1
""")

# (1) 집단별 건수, 제목만 있는 리뷰, 고유 review_id
display(con.execute("""
SELECT late, COUNT(*) AS n_orders, COUNT(msg) AS n_msg,
       CAST(SUM(CASE WHEN title IS NOT NULL AND msg IS NULL THEN 1 ELSE 0 END) AS INTEGER) AS title_only,
       COUNT(DISTINCT CASE WHEN msg IS NOT NULL THEN review_id END) AS uniq_review_msg,
       COUNT(DISTINCT CASE WHEN msg IS NOT NULL OR title IS NOT NULL THEN review_id END) AS uniq_review_msg_or_title
FROM s3_pool GROUP BY late ORDER BY late
""").df())
# 기대: late=0 → 8,247 / 6,449 / 55 / 6,400 / 6,455
#       late=1 → 3,981 / 2,745 / 34 / 2,740 / 2,773

# (2) 본문 단어 수 (짧은 코멘트 비율)
df = con.execute("SELECT late, msg FROM s3_pool WHERE msg IS NOT NULL").df()
df["n_words"] = df.msg.str.split().str.len()
display(df.groupby("late").n_words.agg(
    median="median",
    le2=lambda s: (s <= 2).sum(),
    le3=lambda s: (s <= 3).sum(),
    pct_le3=lambda s: round((s <= 3).mean() * 100, 1)))
# 기대: late=0 → 중앙값 16 / ≤2단어 162 / ≤3단어 278 / 4.3%
#       late=1 → 중앙값 14 / ≤2단어 90 / ≤3단어 169 / 6.2%

# (3) 두 집단에 걸친 review_id
display(con.execute("""
SELECT COUNT(*) AS n_review_id_both_groups FROM (
  SELECT review_id FROM s3_pool WHERE msg IS NOT NULL
  GROUP BY review_id HAVING COUNT(DISTINCT late) > 1)
""").df())
# 기대: 18

con.close()

,late,n_orders,n_msg,title_only,uniq_review_msg,uniq_review_msg_or_title
0,0,8247,6449,55,6400,6455
1,1,3981,2745,34,2740,2773


,median,le2,le3,pct_le3
late,,,,
0,16.0,162,278,4.3
1,14.0,90,169,6.2


,n_review_id_both_groups
0,18
